In [ ]:
import numpy as np
from scipy.stats import logistic
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm
from limvam.caramel import caramel

In [ ]:
# matplotlib style
fontsize = 18
rc = {
    "font.size": fontsize,
    "xtick.labelsize": fontsize,
    "ytick.labelsize": fontsize,
    "font.family": "serif",
}
plt.rcParams.update(rc)

# generate data

In [ ]:
# parameters
m = 5
p = 4
n = 1000
lambda_pen = 1.
random_state = 42
which_noise = "logistic"
shared_causal_ordering = True
use_callback = False

In [ ]:
def generate_data(
    n_views,
    n_components,
    n_samples,
    random_state=None,
    which_noise="logistic",
    shared_causal_ordering=True,
):
    rng = np.random.RandomState(random_state)

    # disturbances S
    if which_noise == "logistic":
        S = logistic.rvs(loc=0, scale=1, size=(n_views, n_components, n_samples), random_state=random_state)
    elif which_noise == "laplace":
        S = rng.laplace(0, 1, size=(n_views, n_components, n_samples))
    elif which_noise == "gaussian":
        S = rng.randn(n_views, n_components, n_samples)
    else:
        raise ValueError("The parameter 'which_noise' should be either 'logistic', 'laplace', or 'gaussian'.")
    
    # causal order P
    if shared_causal_ordering:
        P = np.eye(n_components)[rng.permutation(n_components)]
    else:
        P = np.array([np.eye(n_components)[rng.permutation(n_components)] for _ in range(n_views)])
    
    # triangular matrices T
    T = rng.randn(n_views, n_components, n_components)
    for i in range(n_views):
        T[i][np.triu_indices(n_components, k=0)] = 0

    # adjacency matrices B
    if shared_causal_ordering:
        B = P.T @ T @ P
    else:
        B = np.array([Pi.T @ Ti @ Pi for Pi, Ti in zip(P, T)])

    # mixing matrices A
    A = np.linalg.inv(np.eye(n_components) - B)

    # observations X
    X = np.array([Ai @ Si for Ai, Si in zip(A, S)])

    return X, S, B, T, P

In [ ]:
# generate observations X, causal order P, and causal effects B
X, S, B, T, P = generate_data(m, p, n, random_state, which_noise, shared_causal_ordering)

# CArAMEl

In [ ]:
B_est, T_est, P_est, history = caramel(
    X, lambda_pen=lambda_pen, shared_causal_ordering=shared_causal_ordering, 
    use_callback=use_callback)

In [ ]:
print(f"Shape of B_est : {B_est.shape}")
print(f"Shape of T_est : {T_est.shape}")
print(f"Shape of P_est : {P_est.shape}")

# compute errors on $B^i$, $T^i$, and $P$ (or $P^i$)

In [ ]:
error_B = np.mean((B_est - B) ** 2)
error_T = np.mean((T_est - T) ** 2)
if shared_causal_ordering:
    error_P = 1 - (P_est == P).all()
else:
    error_P = np.mean([1 - (Pe == Pi).all() for Pe, Pi in zip(P_est, P)])

print(f"The average error on B is {error_B:.4f}")
print(f"The average error on T is {error_T:.4f}")
print(f"The error on P is {error_P:.2f}")

# plot

In [ ]:
# plot heat maps of causal effect matrices
def plot_heat_maps(B, title=""):
    m = len(B)
    fig, axes = plt.subplots(1, m, figsize=(m*5, 5))
    norm = TwoSlopeNorm(vmin=np.min(B), vmax=np.max(B), vcenter=0)
    for i, ax in enumerate(axes.flat):
        im = ax.imshow(B[i], norm=norm, cmap="coolwarm")
        ax.set_title(f"View {i}")
    cbar = fig.colorbar(im, ax=axes, fraction=0.0085, pad=0.015)
    cbar.set_label("Color Scale")
    fig.suptitle(f"{title}")
    fig.supxlabel("Variables")
    fig.supylabel("Variables", x=0.1)
    plt.show()

In [ ]:
plot_heat_maps(B, title=r"True adjacency matrices $B^i$")
plot_heat_maps(B_est, title=r"Estimated adjacency matrices $B^i$")

In [ ]:
plot_heat_maps(T, title=r"True strictly lower triangular matrices $T^i$")
plot_heat_maps(T_est, title=r"Estimated strictly lower triangular matrices $T^i$")

In [ ]:
# Plot loss across iterations
if use_callback:
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(history.loss)
    ax.set_yscale('log')
    ax.set_xlim([0, len(history.loss)])
    ax.grid(alpha=0.5)
    ax.set_xlabel("Iterations")
    ax.set_ylabel("Loss")
    ax.set_title(f"CArAMEl's loss across iterations with {which_noise} disturbances")
    plt.show()